# O código abaixo está na ordem necessária para a execução. 
### É usado o Ollama, então será necessário fazer a instalação.

In [11]:
from pathlib import Path
import sqlite3

import logging
import os

In [2]:
from scripts.parser_pdf import parserPdf
from scripts.sqlite import criarBanco
from scripts.embeddings import gerarEmbeddings
from scripts.text_to_sql import textToSQL
from scripts.router import router
from scripts.answer import answer
from scripts.cleanText import cleanText
from scripts.crossEncoder import crossEncoder

c:\Users\ronal\source\repos\ARTEFACT-DesafioTecnico\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PERGUNTA DO USUÁRIO

In [3]:
query = "Qual é o instrumento mais barato e o mais caro?"
#query = "Qual é o prazo de devolução?"
#query = "Qual é o prazo para envio via sedex?"
#query = "Quanto está custando uma bateria com o valor do frete e qual é o prazo para envio via jadlog?"

# PARÂMETROS

In [4]:
pdf_dir = Path("../data/data_pdf")
csv_dir = Path("../data/data_csv")
chunks_dir = Path("../processed_data/chunks")
bd_dir = "../data/dados.db"
embeddings_dir = "../processed_data/embeddings.npy"

k_rank = 5
k_rerank = 1

modelo = "BAAI/bge-m3"
modelo_llm = "qwen2.5-coder:7b"
modelo_rerank = "BAAI/bge-reranker-v2-m3"

temperature = 0.3

conexao = sqlite3.connect(bd_dir)

# CRIAÇÃO DO BANCO DE DADOS, CHUNK DO PDF e GERAÇÃO DOS EMBEDDINGS.
- Só é necessário executar está etapa uma única vez.

In [15]:
#reduz logs do Python
logging.basicConfig(level=logging.WARNING)

parserPdf(pdf_dir, chunks_dir)
criarBanco(csv_dir, conexao)
gerarEmbeddings(modelo, chunks_dir, embeddings_dir)

Encontrados 1 PDF(s).
Processando: politicas_da_loja.pdf
  → 8 chunks criados.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34600.04it/s]


Carregando: politicas_da_loja.jsonl


Batches: 100%|██████████| 1/1 [00:12<00:00, 12.95s/it]


# Roteamento de consulta (Query Routing)

In [16]:
route = router(query, modelo_llm, temperature)
print(route)

SQL


# RANQUEAMENTO
- De acordo com o router, está etapa irá gerar os contextos sendo eles somente de RAG, somente SQL ou ambos.

In [8]:
contextoRAG = ""
resultadoSQL = ""

match route:
    case "RAG":
        print("Executando RAG...")
        resultadoRAG = crossEncoder(chunks_dir, query, modelo, modelo_rerank, embeddings_dir, k_rank=5, k_rerank=1)
        contextoRAG = "\n\n".join(result["text"] for result in resultadoRAG)

    case "SQL":
        print("Executando Text-to-SQL...")
        sql_gerado, cols, registros_sql = textToSQL(query, modelo_llm, temperature, conexao)
        resultadoSQL = f"{cols}\n{registros_sql}"

    case _:
        print("Executando RAG e Text-To-SQL...")
        resultadoRAG = crossEncoder(chunks_dir, query, modelo, modelo_rerank, embeddings_dir, k_rank=5, k_rerank=1)
        contextoRAG = "\n\n".join(result["text"] for result in resultadoRAG)

        sql_gerado, cols, registros_sql = textToSQL(query, modelo_llm, temperature, conexao)
        resultadoSQL = f"{cols}\n{registros_sql}"

Executando Text-to-SQL...


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


# GERA A RESPOSTA FINAL

In [17]:
contextoRAG = cleanText(contextoRAG)
resposta_final = answer(contextoRAG, resultadoSQL, query, modelo_llm, temperature)

# Mostra as resposta
- Resultados do reranking (somente o rank, sem os textos em si)
- Resultados do SQL (sql gerado e os registros gerados)
- Resultado final que o usuário veria

In [18]:
match route:
    case "RAG":
        print("\n\n--- Resultados do reranking ---")
        for result in resultadoRAG:
            print(
                f"\nScore: {result['score']:.4f}"
            )
            print(
                f"Documento: {result['source']}"
            )
            print(
                f"Página: {result['page']}"
            )
            print(
                f"Chunk: {result['chunk_id']}"
            )

    case "SQL":
        print("\n\n--- Resultados do SQL ---")
        print(sql_gerado)
        print("\nColunas:", cols)
        for row in registros_sql:
            print(row)

    case _:
        print("\n\n--- Resultados do reranking ---")
        for result in resultadoRAG:
            print(
                f"\nScore: {result['score']:.4f}"
            )
            print(
                f"Documento: {result['source']}"
            )
            print(
                f"Página: {result['page']}"
            )
            print(
                f"Chunk: {result['chunk_id']}"
            )

        print("\n\n--- Resultados do SQL ---")
        print(sql_gerado)
        print("\nColunas:", cols)
        for row in registros_sql:
            print(row)

print(f"\n--- Resposta final ---\n\n{resposta_final}")



--- Resultados do SQL ---
SELECT * FROM (
    SELECT
        name AS instrumento,
        description AS descricao,
        price_brl AS preco,
        'Mais Barato' AS tipo
    FROM products
    WHERE status = 'active'
    ORDER BY price_brl ASC
    LIMIT 1
)
UNION ALL
SELECT * FROM (
    SELECT
        name AS instrumento,
        description AS descricao,
        price_brl AS preco,
        'Mais Caro' AS tipo
    FROM products
    WHERE status = 'active'
    ORDER BY price_brl DESC
    LIMIT 1
);

Colunas: ['instrumento', 'descricao', 'preco', 'tipo']
('Shelby SU-21S Soprano Sunburst', 'Ukulele soprano com acabamento Sunburst e escala em Rosewood. Visual clássico e som alegre para músicos de todos os níveis.', 159.9, 'Mais Barato')
('Teclado Sintetizador Nord Synth 2 Pro', 'Piano de palco Nord Stage com seções independentes de piano, órgão e synth. Interface intuitiva e qualidade sonora padrão da indústria.', 19567.0, 'Mais Caro')

--- Resposta final ---

Olá! Tudo ótimo aqui na 